In [0]:
%sql
CREATE CATALOG IF NOT EXISTS Data_Integration;
CREATE SCHEMA IF NOT EXISTS Data_Integration.Project;
CREATE VOLUME IF NOT EXISTS Data_Integration.Project.datasets;

In [0]:
from pyspark.sql.functions import col,count,when
from pyspark.sql.types import IntegerType,DateType,StringType
import pyspark.sql.functions as F


In [0]:
display(dbutils.fs.ls("/Volumes/data_integration/project/datasets/Data/"))

path,name,size,modificationTime
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Austin_Calendar.csv,Airbnb_Austin_Calendar.csv,145085718,1781568498000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Austin_Listings.csv,Airbnb_Austin_Listings.csv,27868350,1781371559000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Austin_Reviews.csv,Airbnb_Austin_Reviews.csv,162580204,1781376017000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Bangkok_Calendar.csv,Airbnb_Bangkok_Calendar.csv,398597551,1781568498000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Bangkok_Listings.csv,Airbnb_Bangkok_Listings.csv,61493465,1781371766000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Bangkok_Reviews.csv,Airbnb_Bangkok_Reviews.csv,171237254,1781376834000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Barcelona_Calendar.csv,Airbnb_Barcelona_Calendar.csv,241380886,1781568498000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Barcelona_Listings.csv,Airbnb_Barcelona_Listings.csv,38685344,1781371368000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Barcelona_Reviews.csv,Airbnb_Barcelona_Reviews.csv,308700805,1781385647000
dbfs:/Volumes/data_integration/project/datasets/Data/Airbnb_Berlin_Calendar.csv,Airbnb_Berlin_Calendar.csv,185301369,1781568499000


In [0]:

reviews_dict= {
    "airbnb_austin_reviews_df": "Airbnb_Austin_Reviews.csv",
    "airbnb_berlin_reviews_df":"Airbnb_Berlin_Reviews.csv",
    "airbnb_barcelona_reviews_df": "Airbnb_Barcelona_Reviews.csv",
    "airbnb_bangkok_reviews_df": "Airbnb_Bangkok_Reviews.csv"
}

listings_dict= {
    "airbnb_austin_listings_df": "Airbnb_Austin_Listings.csv",
    "airbnb_berlin_listings_df": "Airbnb_Berlin_Listings.csv",
    "airbnb_barcelona_listings_df": "Airbnb_Barcelona_Listings.csv",
    "airbnb_bangkok_listings_df": "Airbnb_Bangkok_Listings.csv"
}


calendar_dict= {
    "airbnb_austin_calendar_df":  "Airbnb_Austin_Calendar.csv",
    "airbnb_berlin_calendar_df": "Airbnb_Berlin_Calendar.csv",
    "airbnb_barcelona_calendar_df": "Airbnb_Barcelona_Calendar.csv",
    "airbnb_bangkok_calendar_df": "Airbnb_Bangkok_Calendar.csv"
}




In [0]:
"""
reviews_schema_structure = StructType([
    StructField("listings_id", IntegerType()),
    StructField("id", IntegerType()),
    StructField("date", DateType()),
    StructField("reviewer_id", IntegerType()),
    StructField("comments", StringType())
])

def load_files(df,schema_structure):
    Path_Part1="/Volumes/data_integration/project/datasets/Data/"
    for key,values in df.items():
        try: 
            df[key]= spark.read.format("csv").options(delimiter=",", header=True).schema(schema_structure).load(Path_Part1+values)
        except:
            try:
                print("The file has been already loaded,please be aware that you have reran the cell more than once ")
            except:
                print(f"Failed to load the {values} file. Please ensure that the file path is correct")
load_files(reviews_dict,reviews_schema_structure)


"""


'\nreviews_schema_structure = StructType([\n    StructField("listings_id", IntegerType()),\n    StructField("id", IntegerType()),\n    StructField("date", DateType()),\n    StructField("reviewer_id", IntegerType()),\n    StructField("comments", StringType())\n])\n\ndef load_files(df,schema_structure):\n    Path_Part1="/Volumes/data_integration/project/datasets/Data/"\n    for key,values in df.items():\n        try: \n            df[key]= spark.read.format("csv").options(delimiter=",", header=True).schema(schema_structure).load(Path_Part1+values)\n        except:\n            try:\n                print("The file has been already loaded,please be aware that you have reran the cell more than once ")\n            except:\n                print(f"Failed to load the {values} file. Please ensure that the file path is correct")\nload_files(reviews_dict,reviews_schema_structure)\n\n\n'

In [0]:
def load_reviews_files(df):
    Path_Part1="/Volumes/data_integration/project/datasets/Data/"
    for key,values in df.items():
        try: 
            df[key]= spark.read.format("csv").options(delimiter=",", header=True,inferSchema=True,multiline=True).load(Path_Part1+values)
        except:
            print("Failed to load the {values} file. Please ensure that the file path is correct")

load_reviews_files(reviews_dict)

In [0]:
def load_listings_files(df):
    Path_Part1="/Volumes/data_integration/project/datasets/Data/"
    for key,values in df.items():
        try: 
            df[key]= spark.read.format("csv").options(delimiter=",",header=True,inferSchema=True,qoute='""',escape='"',multiline=True).load(Path_Part1+values)
        except:
            print("Failed to load the {values} file. Please ensure that the file path is correct")

load_listings_files(listings_dict)

In [0]:
def load_calendar_files(df):
    Path_Part1="/Volumes/data_integration/project/datasets/Data/"
    for key,values in df.items():
        try: 
            df[key]= spark.read.format("csv").options(delimiter=",", header=True,inferSchema=True).load(Path_Part1+values)
        except:
            print("Failed to load the {values} file. Please ensure that the file path is correct")
load_calendar_files(calendar_dict)

In [0]:
def display_schema(df):
    for key, values in df.items():
        print(f"The schema for {key}: ")
        values.printSchema()

In [0]:
def display_shape_and_dataFrame(df):
    x=0  
    for key,values in df.items():
        x+=1
        print(f"Dataframe: {key} ")
        print(f"The shape of the dataframe is: {(values.count(),len(values.columns))}")
        values.limit(5).display()
        if x>=1 and x<4:
            print("_"*100)

In [0]:
def display_number_of_duplicates_per_dataframe(df):
    duplicate_rows_count=0
    percentage_of_duplicate_rows=0
    x=0
    for key,values in df.items():
        x+=1
        print(f"Dataframe: {key}")
        duplicate_rows_count = values.count() - values.distinct().count()
        percentage_of_duplicate_rows=round(((duplicate_rows_count/values.count())*100),2)
        print(f"The number of duplicate rows: {duplicate_rows_count}")
        print(f"The percentage of duplicate rows: {percentage_of_duplicate_rows}%")
        if x>=1 and x<4:
            print("_"*100)

In [0]:
def display_null_values_per_column(df):
    x=0
    for key,values in df.items():
        x+=1
        print(f'Dataframe: {key}')
        print("The total number of null values per column: ")
        null_count=values.select([count(when(col(c).isNull(), c)).alias(c) for c in values.columns])
        display(null_count)
        
        result_in_percentage=values.select(*[F.round((count(when(col(c).isNull(), c))/(values.count())*100), 2).alias(c) for c in values.columns])                    
        print('The total precentage of null values per column: ')
        display(result_in_percentage)
        if x>=1 and x<4:
            print("_"*100)

In [0]:
def display_description(df):
    x=0
    numerical_columns=[]
    for key,values in df.items():
        x+=1
        print(f'Dataframe: {key}')
        if x<=1:
            for y, t in values.dtypes:
                if t=="int" or t=="double" or t=="bigint":
                    numerical_columns.append(y)
                else:
                    continue 
      
        display(values.describe(numerical_columns))
        if x>=1 and x<4:
            print("_"*100)

In [0]:
display_schema(reviews_dict)

The schema for airbnb_austin_reviews_df: 
root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)

The schema for airbnb_berlin_reviews_df: 
root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)

The schema for airbnb_barcelona_reviews_df: 
root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- reviewer_id: integer (nullable = true)
 |-- reviewer_name: string (nullable = true)
 |-- comments: string (nullable = true)

The schema for airbnb_bangkok_reviews_df: 
root
 |-- listing_id: long (nullable = true)
 |-- id: long (nullable = true)
 |-- date: date (nullable = 

In [0]:
display_shape_and_dataFrame(reviews_dict)

Dataframe: airbnb_austin_reviews_df 
The shape of the dataframe is: (588359, 6)


listing_id,id,date,reviewer_id,reviewer_name,comments
5456,977,2009-03-19,8102,Phil,Highly recommended. Sylvia was extremely helpful and friendly and made our holiday as easy as it could be. The apartment was wonderfully quiet but also only a few minutes' walk to Austin Convention Center.
5456,1039,2009-03-22,8241,Galen,A great place to stay in a great city. Sylvia picked me up at the airport and drove me to Whole Foods so I could buy groceries. She also took me to my first Texas bar-b-que. I saved a ton of money because I didn't have to cab downtown; I could easily walk. The apartment is clean and the bed very comfortable. This place is a lot better than a hotel.
5456,1347,2009-04-08,11152,April,Highly recommended! Cute and cozy guest house ! Thanks Sylvia!!
5456,1491,2009-04-13,12400,Ivonne,"What a great little apartment! It was clean, in a good location and had everything I needed. It was nice not having to spend money on eating out so much b/c there's a refrigerator, microwave, coffeemaker and toaster. Sylvia was extremely kind and very generous - offering to pick us up from the airport. I really enjoyed my stay and recommend Sylvia's place highly!"
5456,1535,2009-04-16,11071,Egan.Sturges.Regan,"""Sylvia was great; """"ditto"""" to all the previous review. She is gracious and generous"


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_reviews_df 
The shape of the dataframe is: (635468, 6)


listing_id,id,date,reviewer_id,reviewer_name,comments
3176,4283,2009-06-20,21475,Milind,"excellent stay, i would highly recommend it. a nice flat in a very nice area. Britta provided clear instructions in securing the keys, etc. Thanks again."
3176,134722,2010-11-07,263467,George,"Britta's apartment in Berlin is in a great area. There are numerous fantastic Restaurants and Bars to suit every taste, easy access to supermarkets and public transportation. The apartment is also surprisingly large. I would surely stay there again."
3176,144064,2010-11-24,76726,Patricia,"Fantastic, large place in good location. Only a short tram ride to Alexanderplatz U-S Bahn, with connections across Berlin. Britta was quick to reply to all messages and gave very clear instructions regarding directions, key pick up and things to know about the place. Try Gargerin cafe around the corner for lovely and cheap continental breakfast. I definitely recommend this apartment!"
3176,156702,2010-12-21,291657,Benedetta,L'appartamento di Britta è molto largo carino confortabile è in un quartiere molto tranquillo e comodo perchè ci sono tanti supermercati e bar dove poter mangiare. Siamo stati benissimo spero di ripetere la bellissima esperienza ... magari in estate! ciao Benedetta
3176,165048,2011-01-04,279789,Aude,"We went in Berlin for the new year eve. The appartment of Britta was really amazing, very chic, cool and in a good district ! You should go there !"


____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_reviews_df 
The shape of the dataframe is: (991785, 6)


listing_id,id,date,reviewer_id,reviewer_name,comments
18674,4808211,2013-05-27,4841196,Caron,"Great location. Clean, spacious flat. Would recommend to anyone."
18674,10660311,2014-03-02,11600277,Juan Carlos,"Mi mejor recomendación para este departamento. Cuida todos los detalles, adicionalmente muy cómodo, limpio, funcional y situado perfectamente."
18674,41087522,2015-08-04,35231385,Shlomi,"Big apartment, well equipped. Very good service, excellent location. Recommended."
18674,81000756,2016-06-20,23223644,Joost,"The Check in was fast and flexible. The price is fair, because the flat ist big enough for 8 people if you are flexible as well. We were the in a time where it wasn't so hot outside, so I have no idea how hot it can get inside. This was my second Airbnb stay so I can't compare it so good with other apartments but I would wish me, that the owner, offers the guests more toilet paper etc. . Because if it is a short stay, you don't want to buy all the staff in big packages. All in all we are very happy with this decision"
18674,278588962,2018-06-18,4756672,Marius,"Great location and enough space in the apartment for 7 people. Although the mattresses were a bit too soft and the house is quite noisy, its a solid choice for a big group of people for a weekend in barcelona."


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_reviews_df 
The shape of the dataframe is: (583313, 6)


listing_id,id,date,reviewer_id,reviewer_name,comments
27934,1094339,2012-04-07,1368195,Michael,"We stayed in the apartment for a week and we enjoyed it very much. Nuttee is a very nice host, and she did her best to accommodate us. Everything is perfect in the apartment, and we love the view from the balcony. The apartment is very modern and spacious, and the location is very central, 10 mins walk to BTS station (and supermarket), or 10 mins by bus/taxi to central world shopping mall. There are also a lot of food stalls and massage nearby. We will definitely stay there again for our next visit to Bangkok. Highly recommended."
27934,1241042,2012-05-07,2007324,Scott,"My girlfriend and I recently stayed in Nuttee's condo for a month. It is a beautiful condo, with a great view, in a great location near Victory Monument. There is an abundance of wonderful street food right around the corner, as well as shopping. This is a perfect area from which to base exploration of Bangkok. Nuttee was out of town, but was still very accessible via email. Her assistant, Suchart, was great, meeting us when we arrived, arranging cleaning of the condo, and seeing us off when we left. He did everything possible to make certain our stay was perfect. We could not have asked for a better experience, and hope to stay here again when we return to Bangkok."
27934,1523384,2012-06-20,2263352,Marc,"""I stayed for one month at the condo and was realy pleased. The condo is at the 19th floor, quiet, modern and clean, and you realy have a """"great city view"""" by day and much more by night. The 24 h Security is very friendly and there is a magnetic entrance system. So you feel save. The communication between Nuttee"
27934,1655571,2012-07-08,558987,Leyla,"Nuttee was a great host! I really enjoyed her apartment and she was absolutely lovely! She even had a bowl of fresh fruit and a sim card for me when I arrived. The apartment itself is really spacious for a single person or couple and has all the mod cons needed to have a relaxing stay in Bangkok. Great air con and the swimming pool was refreshing after a long humid day walking around! I would recommend the apartment as a great, comfortable and relaxing place to stay in Bangkok for sure. The only thing that you should keep in mind is that the BTS (sky train) is a 10 minute walk away - you can take a motorcycle taxi for a few baht or walking is fine in the mornings. There are nice cafes and food places along the street that you walk down to the BTS so its not that bad unless its the middle of a hot day!!"
27934,1972192,2012-08-13,2359865,Rachel,"Nuttee was an amazing host. She and her daughter waited for us at the lobby and gave us a quick rundown of the place. It's a studio apartment, with a classy partition dividing the bedroom from the living room -- and the view from the balcony is awesome. Bed was huge, but if you're coming in a group of 3, the sofa may not always be the most comfortable place for the last guy (bed IS huge enough for three to sleep in though). Shower pressure in the main bathroom was a little low, but we could deal with it. Make sure to ask Nuttee or her assistant to write down the condominium's address for you in Thai, though it will be best if you print out a map yourself -- most of our taxi/tuk-tuk/motorcycle drivers needed a little pointer about where exactly it was. That said, it was a very central location, just slightly away from the sometimes overwhelming hustle and bustle of Pratunam. Warmly recommended."


In [0]:
display_number_of_duplicates_per_dataframe(reviews_dict)

Dataframe: airbnb_austin_reviews_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_berlin_reviews_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_reviews_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_reviews_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%


In [0]:
display_null_values_per_column(reviews_dict)

Dataframe: airbnb_austin_reviews_df
The total number of null values per column: 


listing_id,id,date,reviewer_id,reviewer_name,comments
0,0,0,0,0,0


The total precentage of null values per column: 


listing_id,id,date,reviewer_id,reviewer_name,comments
0.0,0.0,0.0,0.0,0.0,0.0


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_reviews_df
The total number of null values per column: 


...

listing_id,id,date,reviewer_id,reviewer_name,comments
0.0,0.0,0.0,0.0,0.0,0.0


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_reviews_df
The total number of null values per column: 


listing_id,id,date,reviewer_id,reviewer_name,comments
0,0,0,0,5,0


The total precentage of null values per column: 


listing_id,id,date,reviewer_id,reviewer_name,comments
0.0,0.0,0.0,0.0,0.0,0.0


In [0]:
display_description(reviews_dict)

Dataframe: airbnb_austin_reviews_df


summary,listing_id,id,reviewer_id
count,588359,588359,588359
mean,3.037089901847091E17,7.544979264985823E17,1.9024740615397742E8
stddev,4.461337221865626E17,5.329341999897206E17,1.751800123454133E8
min,5456,977,14
max,1507609250285147702,1511780606316276120,718830996


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_reviews_df


summary,listing_id,id,reviewer_id
count,635468,635468,635468
mean,2.018809652772728E17,7.036926587911826E17,1.8695079581661546E8
stddev,4.07001111906846E17,5.5350329280822214E17,1.8348862678392422E8
min,3176,4283,453
max,1510746916580741102,1517346677520075254,719551711


____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_reviews_df


summary,listing_id,id,reviewer_id
count,991785,991785,991785
mean,1.6215047515006992E17,7.45680321447934E17,1.8589527834519577E8
stddev,3.68544760145713E17,5.6233857653724314E17,1.8243507234174562E8
min,18674,110535,3
max,1573973785873262141,1576141575005014467,733213151


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_reviews_df


summary,listing_id,id,reviewer_id
count,583313,583313,583313
mean,3.793460033566031E17,8.093816829252826E17,2.1962561013417155E8
stddev,4.921679986271235E17,5.600653978192871E17,1.9025191343931553E8
min,27934,200452,1
max,1516489022414772324,1519424257819098884,720502542


In [0]:
display_schema(listings_dict)

The schema for airbnb_austin_listings_df: 
root
 |-- id: long (nullable = true)
 |-- listing_url: string (nullable = true)
 |-- scrape_id: long (nullable = true)
 |-- last_scraped: date (nullable = true)
 |-- source: string (nullable = true)
 |-- name: string (nullable = true)
 |-- description: string (nullable = true)
 |-- neighborhood_overview: string (nullable = true)
 |-- picture_url: string (nullable = true)
 |-- host_id: integer (nullable = true)
 |-- host_url: string (nullable = true)
 |-- host_name: string (nullable = true)
 |-- host_since: date (nullable = true)
 |-- host_location: string (nullable = true)
 |-- host_about: string (nullable = true)
 |-- host_response_time: string (nullable = true)
 |-- host_response_rate: string (nullable = true)
 |-- host_acceptance_rate: string (nullable = true)
 |-- host_is_superhost: string (nullable = true)
 |-- host_thumbnail_url: string (nullable = true)
 |-- host_picture_url: string (nullable = true)
 |-- host_neighbourhood: string (nul

In [0]:
display_shape_and_dataFrame(listings_dict)

Dataframe: airbnb_austin_listings_df 
The shape of the dataframe is: (10533, 79)


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
5456,https://www.airbnb.com/rooms/5456,20250916040734,2025-09-17,city scrape,"Walk to 6th, Rainey St and Convention Ctr","Great central location for walking to Convention Center, Rainey Street, East 6th Street, Downtown, Congress Ave Bats. Free wifiNo Smoking, No pets","My neighborhood is ideally located if you want to walk to bars and restaurants downtown, East 6th Street or Rainey Street. The Convention Center is only 3 1/2 blocks away and a quick 10 minute walk. Whole foods store located 5 blks , easily walkable.",https://a0.muscache.com/pictures/14084884/b5a35a84_original.jpg,8028,https://www.airbnb.com/users/show/8028,Sylvia,2009-02-16,"Austin, TX","I am a licensed Real Estate Broker and owner of Armadillo Realty. I attended The University of Texas at Austin and fell in love with the small town that it was back in 1979; I have been here every since. I love the Art, Music and Film scene here in Austin. There is so much natural beauty to enjoy as well. I especially enjoy Barton Springs Pool in the summertime along with the Zilker Hillside theater productions. SXSW, Austin City Limits Festival and the East Austin Art Studio Tour are among my favorite events. I also enjoy a sunset cruise on my canoe to Congress bridge to see the Mexican Freetail Bats come out for their nightly feeding.",within a few hours,100%,90%,t,https://a0.muscache.com/im/users/8028/profile_pic/1329882962/original.jpg?aki_policy=profile_small,https://a0.muscache.com/im/users/8028/profile_pic/1329882962/original.jpg?aki_policy=profile_x_medium,East Downtown,1,2,"['email', 'phone']",t,t,Neighborhood highlights,78702,null,30.26057,-97.73441,Entire guesthouse,Entire home/apt,3,1.0,1 bath,1,2,"[""Iron"", ""Private entrance"", ""Hot water"", ""Dishes and silverware"", ""Refrigerator"", ""HDTV with Amazon Prime Video, HBO Max, Hulu, Netflix, Roku"", ""Self check-in"", ""Keypad"", ""Wifi"", ""Heating"", ""Hair dryer"", ""Bed linens"", ""Long term stays allowed"", ""Patio or balcony"", ""Smoke alarm"", ""Essentials"", ""Luggage dropoff allowed"", ""Air conditioning"", ""Exterior security cameras on property"", ""Shampoo"", ""Extra pillows and blankets"", ""Coffee maker"", ""Microwave"", ""Kitchen"", ""Backyard"", ""Hangers""]",$97.00,2,90,2,4,90,90,2.1,90.0,null,t,13,35,65,328,2025-09-17,708,25,1,81,33,150,14550,2009-03-19,2025-09-02,4.85,4.88,4.86,4.9,4.82,4.73,4.79,null,f,1,1,0,0,3.52
6448,https://www.airbnb.com/rooms/6448,20250916040734,2025-09-17,city scrape,"Secluded Studio @ Zilker - King Bed, Bright & Airy","Clean, private space with everything you need for a quiet, comfy,

____________________________________________________________________________________________________
Dataframe: airbnb_berlin_listings_df 
The shape of the dataframe is: (14274, 79)


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
3176,https://www.airbnb.com/rooms/3176,20250923202926,2025-09-24,city scrape,Fabulous Flat in great Location,"This beautiful first floor apartment is situated at Kollwitzplatz.It is ideal for 2 permanent tenants or a young family with a baby, but can comfortably accommodate 2 additional visitor with the sofa bed in the living room ( arrival Dec 2024)PLEASE INQUIRE FIRST BEFORE MAKING A BOOKING. THANKS","The neighbourhood is famous for its variety of international eateries, pubs, restaurants, cafés, galleries and little shops.The bakery next door is open everyday from up 7 am, which makes warm croissants for breakfast very tempting.On Sundays don't miss the traditional flea markets on Arkonaplatz and in the nearby Mauerpark for special treats and souvenirs. check out our travel guide and please let us know what your highlights were so we can add them for future guests",https://a0.muscache.com/pictures/hosting/Hosting-U3RheVN1cHBseUxpc3Rpbmc6MzE3Ng==/original/725892bf-6973-4121-8ec0-98b234db3657.jpeg,3718,https://www.airbnb.com/users/show/3718,Britta,2008-10-19,"Coledale, Australia",We love to travel ourselves a lot and prefer to stay in apartments. Especially since we had a baby .,within a day,100%,83%,f,https://a0.muscache.com/im/users/3718/profile_pic/1267053077/original.jpg?aki_policy=profile_small,https://a0.muscache.com/im/users/3718/profile_pic/1267053077/original.jpg?aki_policy=profile_x_medium,Prenzlauer Berg,1,1,"['email', 'phone']",t,t,"Berlin, Germany",Prenzlauer Berg Südwest,Pankow,52.53471,13.4181,Entire rental unit,Entire home/apt,2,1.0,1 bath,1,2,"[""Cooking basics"", ""Wine glasses"", ""Crib"", ""Hot water kettle"", ""Baking sheet"", ""Stove"", ""High chair"", ""Books and reading material"", ""Cleaning products"", ""Patio or balcony"", ""Heating"", ""TV"", ""Bed linens"", ""Outdoor dining area"", ""Oven"", ""Toaster"", ""Clothing storage"", ""Fire extinguisher"", ""Kitchen"", ""Host greets you"", ""Iron"", ""Hot water"", ""Wifi"", ""Smoke alarm"", ""Coffee"", ""Outdoor furniture"", ""Essentials"", ""Ethernet connection"", ""Coffee maker"", ""Washer"", ""First aid kit"", ""Long term stays allowed"", ""Extra pillows and blankets"", ""Dining table"", ""Dishes and silverware"", ""Hair dryer"", ""Children\u2019s dinnerware"", ""Bathtub"", ""Hangers"", ""Freezer"", ""Carbon monoxide alarm"", ""Refrigerator"", ""Private hot tub"", ""Board games""]",$105.00,63,730,63,63,730,730,63.0,730.0,null,t,0,0,0,140,2025-09-24,150,2,0,0,0,252,26460,2009-06-20,2025-08-09,4.63,4.67,4.52,4.65,4.7,4.92,4.61,"First name and Last name: Nicolas Krotz Co

____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_listings_df 
The shape of the dataframe is: (18177, 85)


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
18674,https://www.airbnb.com/rooms/18674,20251214012024,2025-12-14,city scrape,Huge flat for 8 people close to Sagrada Familia,"110m2 apartment to rent in Barcelona. Located in the Eixample district, near the Sagrada Familia. It has a small balcony where you can see the temple of Gaudi. Capacity for 8 people. Licence number: HUTB-002062",null,https://a0.muscache.com/pictures/13031453/413cdbfc_original.jpg,71615,https://www.airbnb.com/users/show/71615,1462508358342835057,https://www.airbnb.com/users/profile/1462508358342835057,Mireia,2010-01-19,15,10,14,11,"Barcelona, Spain","We are Mireia (47) & Maria (49), two multilingual entrepreneurs loving Barcelona and having big experience in the touristic market. Ana and Beka from reservation department will attend you and feel youself like at home. The location of our flats perfectly suites for travelling and sightseeing. We are looking forward to sincerely host you in our apartments.",within an hour,95%,90%,f,https://a0.muscache.com/im/pictures/user/User/original/13242c7b-4fc1-415f-a5d7-6474d869a343.jpeg?aki_policy=profile_small,https://a0.muscache.com/im/pictures/user/User/original/13242c7b-4fc1-415f-a5d7-6474d869a343.jpeg?aki_policy=profile_x_medium,la Sagrada Família,39,43,"['email', 'phone']",t,t,null,la Sagrada Família,Eixample,41.40556,2.17262,Entire rental unit,Entire home/apt,8,2.0,2 baths,3,6,"[""Elevator"", ""Refrigerator"", ""Crib"", ""City skyline view"", ""Free washer \u2013 In unit"", ""Essentials"", ""Hangers"", ""Kitchen"", ""Pets allowed"", ""Heating"", ""Coffee maker"", ""Dishes and silverware"", ""Hair dryer"", ""Private patio or balcony"", ""Iron"", ""Shampoo"", ""Wifi"", ""AC - split type ductless system"", ""Host greets you"", ""Hot water"", ""30 inch TV"", ""Pack \u2019n play/Travel crib""]",null,1,1125,1,4,999,999,2.9,999.0,null,t,20,50,74,115,2025-12-14,54,9,1,8,5,54,null,2013-05-27,2025-11-19,4.36,4.42,4.57,4.62,4.62,4.81,4.34,Spain - National registration numberESFCTU000008058000039706000000000000000HUTB-002062349Barcelona - Regional registration numberHUTB-002062,t,24,24,0,0,0.35
2031134,https://www.airbnb.com/rooms/2031134,20251214012024,2025-12-14,city scrape,Sagrada Familia-auditorium. FREE WI-FI.,"Cheap apartment, 45 m2 with a license from the City of Barcelona, 2 bedrooms for a maximum of 4 people with TV, air conditioning and Wi-Fi. Perfect for enjoying your stay in Barcelona. Close to the Sagrada Familia, well connected to the Beach

____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_listings_df 
The shape of the dataframe is: (28806, 79)


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
27934,https://www.airbnb.com/rooms/27934,20250926165947,2025-09-27,city scrape,Nice room with superb city view,"Our cool and comfortable one bedroom apartment in city center with amazing Bangkok view. It comfortably fits two to three persons and is located on a main street, just three blocks from shopping area and only 15 minutes walk to Bangkok downtown.",It is very center of Bangkok and easy access to many interesting places and also other places that close to Bangkok.,https://a0.muscache.com/pictures/566374/23157b6e_original.jpg,120437,https://www.airbnb.com/users/show/120437,Nuttee,2010-05-08,"Bangkok, Thailand","Hi All, I am nuttee patranavik from Bangkok, Thailand. always travel but easy to connect via airbnb..",N/A,N/A,67%,f,https://a0.muscache.com/im/pictures/user/d05a6b51-b459-4eb8-a01e-8c55122b0132.jpg?aki_policy=profile_small,https://a0.muscache.com/im/pictures/user/d05a6b51-b459-4eb8-a01e-8c55122b0132.jpg?aki_policy=profile_x_medium,Victory Monument,1,2,"['email', 'phone']",t,t,"Samsen Nai, Bangkok, Thailand",Ratchathewi,null,13.75983,100.54134,Entire condo,Entire home/apt,2,1.5,1.5 baths,1,1,"[""Free parking on premises"", ""Dryer"", ""Hot water"", ""Gym"", ""Smoke alarm"", ""Host greets you"", ""Luggage dropoff allowed"", ""Essentials"", ""Kitchen"", ""Elevator"", ""Hair dryer"", ""Fire extinguisher"", ""Dishes and silverware"", ""Microwave"", ""Washer"", ""Refrigerator"", ""Wifi"", ""Hangers"", ""Stove"", ""Iron"", ""Long term stays allowed"", ""Air conditioning"", ""Pool"", ""Shampoo"", ""Patio or balcony""]","$1,595.00",15,240,30,30,240,240,30.0,240.0,null,t,27,57,87,362,2025-09-27,65,0,0,93,1,0,0,2012-04-07,2024-09-17,4.86,4.95,4.82,4.97,4.91,4.66,4.75,null,f,1,1,0,0,0.4
27979,https://www.airbnb.com/rooms/27979,20250926165947,2025-09-27,previous scrape,"Easy going landlord,easy place",null,null,https://a0.muscache.com/pictures/106247594/1d6bc6c7_original.jpg,120541,https://www.airbnb.com/users/show/120541,Emy,2010-05-08,"Bangkok, Thailand",null,N/A,N/A,N/A,f,https://a0.muscache.com/im/users/120541/profile_pic/1436431885/original.jpg?aki_policy=profile_small,https://a0.muscache.com/im/users/120541/profile_pic/1436431885/original.jpg?aki_policy=profile_x_medium,null,2,4,"['email', 'phone']",t,f,null,Bang Na,null,13.66818,100.61674,Private room in rental unit,Private room,2,null,1 bath,null,null,"[""Essentials"", ""Heating"", ""Free parking on premises"", ""First aid kit"", ""Kitchen"", ""Elevator"", ""Hot tub"", ""Wifi"", ""TV with standard cable"", ""Fire extinguisher"", ""Washer"", ""Pool"", ""Smoke alarm"", ""Shampoo"", ""G

In [0]:
display_number_of_duplicates_per_dataframe(listings_dict)

Dataframe: airbnb_austin_listings_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_berlin_listings_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_listings_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_listings_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%


In [0]:
display_null_values_per_column(listings_dict)

Dataframe: airbnb_austin_listings_df
The total number of null values per column: 


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,0,0,0,0,0,222,5173,0,0,0,9,9,1875,3575,9,9,9,400,9,9,836,9,9,0,9,9,5173,0,10533,0,0,0,0,0,6,32,13,9,0,16,0,0,6,6,6,6,0,0,10533,85,0,0,0,0,0,0,0,0,0,0,0,16,1624,1624,1624,1624,1624,1624,1624,1624,1624,10533,0,0,0,0,0,1624


The total precentage of null values per column: 


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0.0,0.0,0.0,0.0,0.0,0.0,2.11,49.11,0.0,0.0,0.0,0.09,0.09,17.8,33.94,0.09,0.09,0.09,3.8,0.09,0.09,7.94,0.09,0.09,0.0,0.09,0.09,49.11,0.0,100.0,0.0,0.0,0.0,0.0,0.0,0.06,0.3,0.12,0.09,0.0,0.15,0.0,0.0,0.06,0.06,0.06,0.06,0.0,0.0,100.0,0.81,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.15,15.42,15.42,15.42,15.42,15.42,15.42,15.42,15.42,15.42,100.0,0.0,0.0,0.0,0.0,0.0,15.42


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_listings_df
The total number of null values per column: 


...

id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_profile_id,host_profile_url,host_name,host_since,hosts_time_as_user_years,hosts_time_as_user_months,hosts_time_as_host_years,hosts_time_as_host_months,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0.0,0.0,0.0,0.0,0.0,0.0,3.45,100.0,0.0,0.0,0.0,0.0,0.0,0.03,0.03,0.03,0.03,0.03,0.03,23.45,36.08,0.03,0.03,0.03,0.0,0.03,0.03,53.86,0.03,0.03,0.0,0.03,0.03,100.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.57,0.05,9.07,19.98,0.0,100.0,0.0,0.0,0.04,0.04,0.04,0.04,0.0,0.0,100.0,5.63,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,27.96,27.96,27.96,27.97,27.97,27.98,27.96,27.97,27.97,39.41,0.0,0.0,0.0,0.0,0.0,27.96


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_listings_df
The total number of null values per column: 


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0,0,0,0,0,0,776,19322,0,0,0,9,9,7394,11990,9,9,9,1610,9,9,19436,9,9,0,9,9,19322,0,28806,0,0,0,0,0,5589,122,1304,5582,0,5533,0,0,3,3,3,3,0,0,28806,2248,0,0,0,0,0,0,0,0,0,0,0,5533,10090,10090,10090,10090,10091,10093,10091,10094,10096,28806,0,0,0,0,0,10090


The total precentage of null values per column: 


id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
0.0,0.0,0.0,0.0,0.0,0.0,2.69,67.08,0.0,0.0,0.0,0.03,0.03,25.67,41.62,0.03,0.03,0.03,5.59,0.03,0.03,67.47,0.03,0.03,0.0,0.03,0.03,67.08,0.0,100.0,0.0,0.0,0.0,0.0,0.0,19.4,0.42,4.53,19.38,0.0,19.21,0.0,0.0,0.01,0.01,0.01,0.01,0.0,0.0,100.0,7.8,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,19.21,35.03,35.03,35.03,35.03,35.03,35.04,35.03,35.04,35.05,100.0,0.0,0.0,0.0,0.0,0.0,35.03


In [0]:
display_description(listings_dict)

Dataframe: airbnb_austin_listings_df


summary,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,10533,10533,10533,10524,10524,10533,10533,10533,10533,10527,10520,10524,10533,10533,10527,10527,10527,10527,10533,10533,10533,10533,10533,10533,10533,10533,10533,10533,10533,10533,10517,8909,8909,8909,8909,8909,8909,8909,10533,10533,10533,10533,8909
mean,7.673743593253303E17,2.025091604073583E13,2.0513087526431215E8,114.13683010262258,165.26881413911062,78724.7826829963,30.281656347298483,-97.74922485143652,5.334187790752872,1.7374845635033722,2.116730038022814,2.953724819460281,7.768157220165195,430.7249596506218,6.902156359836611,11.972926759760616,608.9635223710459,8568543.45616035,8.30261084211539,8469802.461891212,15.66495775182759,35.0497484097598,57.74964397607519,234.90923763410234,55.858919586062854,13.065698281591189,0.867559099971518,69.69097123326688,11.793885882464634,82.12835849235735,13786.05952267757,4.840634190144789,4.8591862161858606,4.820099898978534,4.894308003142865,4.902264002693903,4.825560668986411,4.778510495005024,9.856735972657363,8.208677489793981,1.0055065033703598,0.5205544479255673,1.7428375799753084
stddev,5.572353002213585E17,1.5026542653046966,2.1250703311582574E8,563.9942573122173,787.6561387817876,20.86811251996888,0.06563145099982233,0.06493406867659306,3.636148780052719,1.0605051420444092,1.550881656133892,3.0052136265312193,20.586237793069706,389.5635737971843,19.488293404174645,38.89726805208635,455.3027474846141,1.3538002925215825E8,20.371346221736395,1.338584209853515E8,9.858371527130597,18.442255297623216,26.98986110848355,114.1024666061331,104.27686639255637,18.844387787386044,1.5750091163916173,31.42074336323566,18.786170394384854,86.395396259338,31745.411148946263,0.3040074202105453,0.2889672784561189,0.31784280241412605,0.27754348090753617,0.26441844140255677,0.28433651261997916,0.32833192978232306,17.80888971403955,15.960759796844819,4.483836869346121,5.55318859111474,1.8095631763800175
min,5456,20250916040734,23,1,1,78701,30.07844,-98.05335,1,0.0,0,0,1,1,1,1,1,1,1.0,1.0,0,0,0,0,0,0,0,0,0,0,0,1.0,1.0,1.0,1.0,1.0,1.0,1.0,1,0,0,0,0.01
max,1510543330054750084,20250916040734,718142318,5030,9684,78759,30.5194,-97.56244,16,17.0,23,132,365,1825,365,915,1825,2147483647,365.0,2.123949592E9,30,60,90,365,1305,396,25,107,217,255,2400000,5.0,5.0,5.0,5.0,5.0,5.0,5.0,96,96,42,60,41.14


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_listings_df


summary,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,14274,14274,14274,14260,14260,14274,14274,14274,14274,9330,12250,9279,14274,14274,14269,14269,14269,14269,14274,14274,14274,14274,14274,14274,14274,14274,14274,14274,14274,14274,9264,10960,10958,10960,10957,10959,10957,10956,14274,14274,14274,14274,10960
mean,5.216123118308992E17,2.0250923202919945E13,1.9169577812372145E8,25.74796633941094,29.718373071528752,null,52.5089178660555,13.402306612844772,3.0529634300126105,1.1478563772775991,1.3568979591836734,2.0257570858928764,39.965531736023536,535.8365559759002,39.309552176045976,44.34263087812741,301613.37697105616,301645.7472843227,40.08449628695514,301526.4439470365,8.306221101303068,20.597940311055066,34.19672131147541,146.8295502311896,44.519475970295645,9.4640605296343,0.7533277287375648,37.92307692307692,8.511909766008127,77.2503853159591,14309.615716753022,4.754666058394144,4.794140354079192,4.698031934306552,4.826760974719348,4.820637831918973,4.76111435611934,4.647220700985737,14.070898136471907,11.890219980383915,1.9739386296763346,0.1445285133809724,1.2984552919708072
stddev,5.756023336460658E17,1.515309575167801,2.0979307864217627E8,106.6502918226023,126.14575257696194,null,0.0338244944011059,0.06813727195064208,1.9250286438603048,0.48043310037357967,0.8342826883532773,1.670110677851671,53.50247948964854,461.9740077791996,52.901565887956025,63.593882858624696,2.5423348485688563E7,2.5423348102408413E7,52.951325052788405,2.541889545133266E7,10.373702894004476,22.180652243529128,34.62480141280057,140.46748781539588,100.53872999118674,21.018333426231305,1.7385807013892451,38.04515006485038,20.66289581492058,101.28349829787727,29001.050971747012,0.36234917258602345,0.34514616994654196,0.4045216024824745,0.3244645864945555,0.3668549016296342,0.3286952274488102,0.40445701161456105,48.44048994100398,46.97473049151933,11.31793207409567,1.527689677650939,2.08390967368253
min,3176,20250923202926,1581,1,1,Adlershof,52.34171613270607,13.1168935,1,0.0,0,0,1,1,1,1,1,1,1.0,1.0,0,0,0,0,0,0,0,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0.01
max,1516054633356759428,20250923202926,719270124,1359,2112,südliche Luisenstadt,52.65611,13.72139,16,15.0,14,50,1125,9999,1125,1125,2147483647,2147483647,1125.0,2.147483647E9,30,60,90,365,2895,684,42,100,639,255,1900000,5.0,5.0,5.0,5.0,5.0,5.0,5.0,311,311,114,24,48.87


____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_listings_df


summary,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,18177,18177,18177,18171,18171,18177,18177,18177,18177,14619,16528,14545,18177,18177,18169,18169,18169,18169,18177,18177,18177,18177,18177,18177,18177,18177,18177,18177,18177,18177,0,13095,13092,13093,13091,13094,13092,13092,18177,18177,18177,18177,13095
mean,6.447131137703437E17,2.0251214012030434E13,2.0954166976354733E8,94.11215673325628,126.72092895272688,null,41.392061889828284,2.1668414290899003,3.5308906860317983,1.4620015048908954,1.921043078412391,2.655895496734273,16.592507014358805,508.6518677449524,15.417194121855909,23.7681215256756,523.4671143155925,236982.58957565084,19.88331407823071,225189.7323650766,12.411178962425042,27.899708422732022,44.98712658854596,209.57215162017934,54.563184243824615,11.200638169114816,0.7466578643340486,6.66688672498212,9.59905374924355,78.75309457006107,null,4.596749140893446,4.650277268560938,4.617474986634059,4.71405851348255,4.707525584237044,4.75423311946224,4.455212343415828,68.49023491225175,52.902129064202015,15.38345161467789,0.10210705837046817,1.4233776250477308
stddev,6.114387163691318E17,1.041798121907056,2.1661721355742645E8,197.1339483300162,251.73810501884049,null,0.013634545764569104,0.01761964932259789,2.3906188711579484,0.9064401403315681,1.451166768241891,2.5058616233397175,31.589898597683657,391.3993645039053,28.388050684241573,60.90489844287267,422.11553320246963,2.2530315764013004E7,50.1142727235423,2.1414518616253223E7,11.062017198312189,22.258623319623158,33.432109269635724,130.31369578924748,116.77927928610393,24.999374191768116,2.171639679282943,6.754261496793311,22.1295848273918,96.66573999487134,null,0.5316326510071674,0.5104423359347259,0.5228663686484825,0.5003182693065337,0.5210213011794164,0.3881909334184426,0.5817734175336753,129.19511855944634,122.64717483194009,54.5747682865465,0.7260134861473339,2.2285783075589403
min,18674,20251214012024,3073,1,1,Can Baró,41.35067283884408,2.09276,1,0.0,0,0,1,1,1,1,1,1,1.0,1.0,0,0,0,0,0,0,0,0,0,0,null,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0.01
max,1574760542225568017,20251214012024,732040813,2874,3807,les Tres Torres,41.462243,2.22183,16,16.0,30,127,1124,1125,1124,1124,1125,2147483647,1124.0,2.0415803993E9,30,60,90,365,2042,938,112,18,885,255,null,5.0,5.0,5.0,5.0,5.0,5.0,5.0,549,549,316,12,99.79


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_listings_df


summary,id,scrape_id,host_id,host_listings_count,host_total_listings_count,neighbourhood_cleansed,latitude,longitude,accommodates,bathrooms,bedrooms,beds,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,availability_30,availability_60,availability_90,availability_365,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month
count,28806,28806,28806,28797,28797,28806,28806,28806,28806,23217,27502,23224,28806,28806,28803,28803,28803,28803,28806,28806,28806,28806,28806,28806,28806,28806,28806,28806,28806,28806,23273,18716,18716,18715,18713,18715,18712,18710,28806,28806,28806,28806,18716
mean,7.995041352921178E17,2.0250926165950484E13,2.855787045057974E8,49.95121019550648,61.52738132444352,null,13.744811129336366,100.56273509607188,3.137540790113171,1.3985010983331179,1.4179696022107484,1.8209610747502583,14.107373463861695,489.5243004929529,13.008054716522585,14.191993889525397,75200.33840224976,597111.3519077874,14.007140873429158,595616.0975595356,18.92255085746025,40.468687079080745,62.68346872179407,250.04846212594597,20.250399222384225,5.839859751440672,0.3409012011386517,66.57224189404985,5.167985836284108,53.158543359022424,109371.32110170583,4.690126095319539,4.721540927548625,4.6817013091103385,4.764033559557542,4.7808346246326545,4.659067977768301,4.646764831640842,25.863917239464,20.40567937235298,5.077796292439075,0.20912309935430118,0.9709136567642639
stddev,5.786117159678876E17,0.32031459421552094,2.3096968045085534E8,130.95367420780363,158.92163774863565,null,0.04095170762737208,0.0499988256600052,2.2758329237770205,1.0455071874651412,1.406303494117188,1.8351465412192283,43.24329948863744,897.3076545545052,39.65304712046701,41.01742321858141,1.2653505971377151E7,3.578516991634671E7,40.69874030768375,3.5697539793022126E7,11.887022721487028,22.687451907949452,33.23816727057195,126.34739945056752,54.79959444948583,16.500543902323376,1.2057311242546882,35.191779568979996,15.771139332521235,81.80409215030322,432327.89845834055,0.5384677354612667,0.5190425102235414,0.518982860905175,0.50998447794936,0.5073884567242912,0.4808894579946016,0.5433060532420012,41.23798886449929,38.104656415046634,18.564434960206174,2.7887382473009854,1.4758261718346253
min,27934,20250926165947,21447,0,0,Bang Bon,13.52937,100.3289236538978,1,0.0,0,0,1,1,1,1,1,1,1.0,1.0,0,0,0,0,0,0,0,0,0,0,0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1,0,0,0,0.01
max,1518168294097881222,20250926165947,720639377,1455,8774,Yan na wa,13.95354,100.92371,16,50.0,50,50,1115,100000,1115,1115,2147483647,2147483647,1115.0,2.147483647E9,30,60,90,365,2926,1304,85,96,795,255,57170256,5.0,5.0,5.0,5.0,5.0,5.0,5.0,243,234,193,58,57.6


In [0]:
display_schema(calendar_dict)

The schema for airbnb_austin_calendar_df: 
root
 |-- listing_id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- available: string (nullable = true)
 |-- price: string (nullable = true)
 |-- adjusted_price: string (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- maximum_nights: integer (nullable = true)

The schema for airbnb_berlin_calendar_df: 
root
 |-- listing_id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- available: string (nullable = true)
 |-- price: string (nullable = true)
 |-- adjusted_price: string (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- maximum_nights: integer (nullable = true)

The schema for airbnb_barcelona_calendar_df: 
root
 |-- listing_id: long (nullable = true)
 |-- date: date (nullable = true)
 |-- available: string (nullable = true)
 |-- price: string (nullable = true)
 |-- adjusted_price: string (nullable = true)
 |-- minimum_nights: integer (nullable = true)
 |-- maximum_nights: i

In [0]:
display_shape_and_dataFrame(calendar_dict)

Dataframe: airbnb_austin_calendar_df 
The shape of the dataframe is: (3844547, 7)


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
5456,2025-09-17,f,null,null,4,90
5456,2025-09-18,f,null,null,4,90
5456,2025-09-19,f,null,null,4,90
5456,2025-09-20,f,null,null,4,90
5456,2025-09-21,t,null,null,4,90


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_calendar_df 
The shape of the dataframe is: (5210011, 7)


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
1358910,2025-09-24,f,null,null,3,365
1358910,2025-09-25,f,null,null,3,365
1358910,2025-09-26,t,null,null,3,365
1358910,2025-09-27,t,null,null,3,365
1358910,2025-09-28,t,null,null,3,365


____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_calendar_df 
The shape of the dataframe is: (6634623, 7)


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
660131950555612074,2025-12-14,f,null,null,1,999
660131950555612074,2025-12-15,t,null,null,1,999
660131950555612074,2025-12-16,t,null,null,1,999
660131950555612074,2025-12-17,t,null,null,1,999
660131950555612074,2025-12-18,t,null,null,1,999


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_calendar_df 
The shape of the dataframe is: (10514202, 7)


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
5675191,2025-09-27,f,null,null,4,1125
1920431,2025-09-26,f,null,null,1,31
1920431,2025-09-27,t,null,null,1,31
1920431,2025-09-28,f,null,null,1,31
1920431,2025-09-29,t,null,null,1,31


In [0]:
display_number_of_duplicates_per_dataframe(calendar_dict)

Dataframe: airbnb_austin_calendar_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_berlin_calendar_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_calendar_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%
____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_calendar_df
The number of duplicate rows: 0
The percentage of duplicate rows: 0.0%


In [0]:
display_null_values_per_column(calendar_dict)

Dataframe: airbnb_austin_calendar_df
The total number of null values per column: 


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,0,0,3844547,3844547,0,0


The total precentage of null values per column: 


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0.0,0.0,0.0,100.0,100.0,0.0,0.0


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_calendar_df
The total number of null values per column: 


...

listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0.0,0.0,0.0,100.0,100.0,0.0,0.0


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_calendar_df
The total number of null values per column: 


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0,0,0,10514202,10514202,0,0


The total precentage of null values per column: 


listing_id,date,available,price,adjusted_price,minimum_nights,maximum_nights
0.0,0.0,0.0,100.0,100.0,0.0,0.0


In [0]:
display_description(calendar_dict)

Dataframe: airbnb_austin_calendar_df


summary,listing_id,minimum_nights,maximum_nights
count,3844547,3844547,3844547
mean,7.673739601239629E17,8.30339517243514,8469798.056218848
stddev,5.572090501313459E17,20.950577824197513,1.3459452159376153E8
min,5456,1,1
max,1510543330054750084,915,2147483647


____________________________________________________________________________________________________
Dataframe: airbnb_berlin_calendar_df


summary,listing_id,minimum_nights,maximum_nights
count,5210011,5210011,5210011
mean,5.216122117128339E17,40.08477275767748,301526.38596060546
stddev,5.755822160427651E17,53.1359281769399,2.5418005044701558E7
min,3176,1,1
max,1516054633356759428,1125,2147483647


____________________________________________________________________________________________________
Dataframe: airbnb_barcelona_calendar_df


summary,listing_id,minimum_nights,maximum_nights
count,6634623,6634623,6634623
mean,6.447122938031841E17,19.883100818237903,225189.1309661453
stddev,6.114220205457143E17,51.8681195935604,2.1962343536665626E7
min,18674,1,1
max,1574760542225568017,1124,2147483647


____________________________________________________________________________________________________
Dataframe: airbnb_bangkok_calendar_df


summary,listing_id,minimum_nights,maximum_nights
count,10514202,10514202,10514202
mean,7.995040159426332E17,14.007109431604984,595615.4186006698
stddev,5.78601760325162E17,40.828157913306036,3.573976311039628E7
min,27934,1,1
max,1518168294097881222,1115,2147483647


In [0]:
def save_to_table(df):
    for key, values in df.items():
        try:
            values.write.mode("overwrite").saveAsTable(key) 
        except:
             print(f"Failed to save {key} as a table")

In [0]:
reviews_raw_dfs_dict = {
    "data_integration.project.airbnb_austin_reviews_df_raw": reviews_dict["airbnb_austin_reviews_df"],
    "data_integration.project.airbnb_berlin_reviews_df_raw": reviews_dict["airbnb_berlin_reviews_df"],
    "data_integration.project.airbnb_barcelona_reviews_df_raw": reviews_dict["airbnb_barcelona_reviews_df"],
    "data_integration.project.airbnb_bangkok_reviews_df_raw": reviews_dict["airbnb_bangkok_reviews_df"]
}

listings_raw_dfs_dict = {
    "data_integration.project.airbnb_austin_listings_df_raw": listings_dict["airbnb_austin_listings_df"],
    "data_integration.project.airbnb_berlin_listings_df_raw": listings_dict["airbnb_berlin_listings_df"],
    "data_integration.project.airbnb_barcelona_listings_df_raw": listings_dict["airbnb_barcelona_listings_df"],
    "data_integration.project.airbnb_bangkok_listings_df_raw": listings_dict["airbnb_bangkok_listings_df"]
}

calendar_raw_dfs_dict = {
    "data_integration.project.airbnb_austin_calendar_df_raw": calendar_dict["airbnb_austin_calendar_df"],
    "data_integration.project.airbnb_berlin_calendar_df_raw": calendar_dict["airbnb_berlin_calendar_df"],
    "data_integration.project.airbnb_barcelona_calendar_df_raw": calendar_dict["airbnb_barcelona_calendar_df"],
    "data_integration.project.airbnb_bangkok_calendar_df_raw": calendar_dict["airbnb_bangkok_calendar_df"]
}



In [0]:
%sql
DROP TABLE IF EXISTS data_integration.project.airbnb_austin_reviews_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_berlin_reviews_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_barcelona_reviews_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_bangkok_reviews_df_raw;

DROP TABLE IF EXISTS data_integration.project.airbnb_austin_listings_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_berlin_listings_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_barcelona_listings_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_bangkok_listings_df_raw;

DROP TABLE IF EXISTS data_integration.project.airbnb_austin_calender_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_berlin_calender_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_barcelona_calender_df_raw;
DROP TABLE IF EXISTS data_integration.project.airbnb_bangkok_calender_df_raw;

In [0]:
save_to_table(reviews_raw_dfs_dict)
save_to_table(listings_raw_dfs_dict)
save_to_table(calendar_raw_dfs_dict)